In [21]:
# Attach macro features at each company-event prediction date.
# Keep unavailable source series missing and use only observations dated on or before prediction.

In [22]:
import sys as _sys
from pathlib import Path as _Path
_UTILS_ROOT = next(path for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
                   if (path / "notebook_utils.py").is_file())
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths
from project_config import MACRO_FIELDS
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/market_features.csv").exists():
    ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
MAX_AGE_DAYS = {"D": 10, "M": 62, "Q": 120}
MIN_YEAR_OBSERVATIONS = {"D": 200, "M": 12, "Q": 4}
SUFFIXES = ["level_at_prediction_date", "change_1y", "change_2y", "change_3y", "average_1y", "volatility_1y"]

market_features = pd.read_csv(PROCESSED / "market_features.csv")
macro_long = pd.read_csv(PROCESSED / "clean_macro_long.csv", usecols=["date", "field", "value"])
if market_features.empty or market_features[["event_id", "company_id", "prediction_date"]].isna().any().any():
    raise ValueError("Missing company-events or identifiers")
if market_features.duplicated(["company_id", "prediction_date"]).any() or not market_features["event_id"].is_unique:
    raise ValueError("Input must contain unique company-events")
event_dates = pd.to_datetime(market_features["prediction_date"], errors="raise").dt.normalize().astype("datetime64[ns]")
macro_long["date"] = pd.to_datetime(macro_long["date"], errors="raise").dt.normalize().astype("datetime64[ns]")
macro_long["value"] = pd.to_numeric(macro_long["value"], errors="raise")
if macro_long["date"].isna().any() or macro_long.duplicated(["date", "field"]).any():
    raise ValueError("Missing dates or duplicate macro records")
if not np.isfinite(macro_long.loc[macro_long["value"].notna(), "value"]).all():
    raise ValueError("Nonfinite cleaned macro values")

credit_metadata = MACRO_FIELDS.get("credit_spread", {})
if "credit_spread" in set(macro_long["field"]) and not credit_metadata.get("security"):
    raise ValueError("Collected credit_spread requires an identifiable source mapping")
credit_is_total_return_index = (str(credit_metadata.get("security", "")).strip().upper() == "LUACTRUU INDEX"
                              and str(credit_metadata.get("field", "")).strip().upper() == "PX_LAST")

VARIABLES = {
    "inflation": {"source": "inflation_cpi_yoy", "frequency": "M", "alias": "inflation_level"},
    "gdp_growth": {"source": "real_gdp_growth", "frequency": "Q", "alias": "gdp_growth"},
    "unemployment_rate": {"source": "unemployment_rate", "frequency": "M", "alias": "unemployment_rate"},
    "federal_funds_rate": {"source": "effective_federal_funds_rate", "frequency": "D", "alias": "federal_funds_rate"},
    "treasury_10y_yield": {"source": "treasury_10y_rate", "frequency": "D", "alias": "treasury_10y_yield"},
    "treasury_3m_yield": {"source": "treasury_3m_rate", "frequency": "D", "alias": "treasury_3m_yield"},
    "yield_curve_spread": {"source": "yield_curve_spread", "frequency": "D", "alias": "yield_curve_spread"},
    "credit_spread": {"source": None if credit_is_total_return_index else "credit_spread", "frequency": "M", "alias": "credit_spread"},
    "vix": {"source": "vix", "frequency": "D", "alias": "vix"},
    "recession_indicator": {"source": "recession_indicator", "frequency": "M", "alias": "recession_indicator"},
}
if credit_is_total_return_index:
    VARIABLES["corporate_bond_total_return_index"] = {"source": "credit_spread", "frequency": "M", "alias": "corporate_bond_total_return_index"}
for specification in VARIABLES.values():
    configured = MACRO_FIELDS.get(specification["source"], {}).get("periodicity")
    if configured is not None:
        specification["frequency"] = str(configured).strip().upper()
    if specification["frequency"] not in MAX_AGE_DAYS:
        raise ValueError("Unsupported macro source frequency")


In [23]:
# Construct spreads only from yields observed on exactly the same source date.
def build_yield_curve_source(macro_history):
    yields = macro_history.loc[macro_history["field"].isin(["treasury_10y_rate", "treasury_3m_rate"])].pivot(
        index="date", columns="field", values="value")
    if not {"treasury_10y_rate", "treasury_3m_rate"}.issubset(yields.columns):
        raise ValueError("Both 10-year and 3-month yields are required for the yield curve")
    curve = (yields["treasury_10y_rate"] - yields["treasury_3m_rate"]).rename("value").reset_index()
    curve["field"] = "yield_curve_spread"
    return curve

yield_curve = build_yield_curve_source(macro_long)
source_rows = pd.concat([macro_long, yield_curve[["date", "field", "value"]]], ignore_index=True)
if source_rows.duplicated(["date", "field"]).any():
    raise ValueError("Derived spread conflicts with an existing source field")
histories = {}
for field, group in source_rows.loc[source_rows["value"].notna()].groupby("field", sort=False):
    group = group.sort_values("date")
    histories[field] = (group["date"].to_numpy(dtype="datetime64[ns]"), group["value"].to_numpy(dtype=float))
if "recession_indicator" in histories:
    if not np.isin(histories["recession_indicator"][1], [0, 1]).all():
        raise ValueError("Collected recession indicator must contain 0/1 values")


In [24]:
def macro_endpoint(dates, values, cutoff, max_age_days):
    position = np.searchsorted(dates, cutoff, side="right") - 1
    if position < 0 or dates[position] < cutoff - np.timedelta64(max_age_days, "D"):
        return np.nan, np.datetime64("NaT", "ns")
    return float(values[position]), dates[position]


def annual_macro_statistics(dates, values, prediction, frequency, max_age_days, minimum_observations):
    cutoff = (pd.Timestamp(prediction) - pd.DateOffset(months=12)).to_datetime64()
    start = np.searchsorted(dates, cutoff, side="right")
    stop = np.searchsorted(dates, prediction, side="right")
    observed_dates = dates[start:stop]
    observed_values = values[start:stop]
    count = len(observed_values)
    first = observed_dates[0] if count else np.datetime64("NaT", "ns")
    last = observed_dates[-1] if count else np.datetime64("NaT", "ns")
    complete = count >= minimum_observations
    if count:
        complete = (complete and first <= cutoff + np.timedelta64(max_age_days, "D")
                    and last >= prediction - np.timedelta64(max_age_days, "D"))
        if frequency in ["M", "Q"]:
            periods = observed_dates.astype("datetime64[M]").astype(np.int64)
            if frequency == "Q":
                periods = periods // 3
            periods = np.unique(periods)
            complete = complete and len(periods) >= minimum_observations and (np.diff(periods) == 1).all()
        else:
            gaps = np.diff(observed_dates).astype("timedelta64[D]").astype(int)
            complete = complete and (gaps <= max_age_days).all()
    average = np.nan
    volatility = np.nan
    if complete:
        with np.errstate(over="ignore", invalid="ignore"):
            average = np.mean(observed_values)
            volatility = np.std(observed_values, ddof=1)
        if not np.isfinite(average):
            average = np.nan
        if not np.isfinite(volatility):
            volatility = np.nan
    return average, volatility, first, last, count, bool(complete)


def calculate_macro_features(prediction_dates, source_histories, variable_definitions, age_limits, minimum_counts):
    records = []
    audits = []
    empty_dates = np.array([], dtype="datetime64[ns]")
    empty_values = np.array([], dtype=float)
    for date in prediction_dates:
        prediction = pd.Timestamp(date).to_datetime64()
        record = {}
        audit = {}
        for variable, specification in variable_definitions.items():
            source = specification["source"]
            frequency = specification["frequency"]
            age_limit = age_limits[frequency]
            dates, values = source_histories.get(source, (empty_dates, empty_values))
            level, reference_date = macro_endpoint(dates, values, prediction, age_limit)
            record[f"{variable}_level_at_prediction_date"] = level
            record[specification["alias"]] = level
            audit[f"{variable}_macro_reference_date"] = reference_date
            audit[f"{variable}_macro_days_before_prediction"] = (
                int((prediction - reference_date) / np.timedelta64(1, "D")) if not np.isnat(reference_date) else np.nan)
            for lag in [1, 2, 3]:
                cutoff = (pd.Timestamp(date) - pd.DateOffset(years=lag)).to_datetime64()
                prior, prior_date = macro_endpoint(dates, values, cutoff, age_limit)
                change = np.nan
                if np.isfinite(level) and np.isfinite(prior):
                    with np.errstate(over="ignore", invalid="ignore"):
                        change = np.subtract(level, prior)
                    if not np.isfinite(change):
                        change = np.nan
                record[f"{variable}_change_{lag}y"] = change
                audit[f"{variable}_macro_reference_date_{lag}y"] = prior_date
                audit[f"{variable}_macro_lag_cutoff_date_{lag}y"] = cutoff
            average, volatility, first, last, count, complete = annual_macro_statistics(
                dates, values, prediction, frequency, age_limit, minimum_counts[frequency])
            record[f"{variable}_average_1y"] = average
            record[f"{variable}_volatility_1y"] = volatility
            audit[f"{variable}_macro_window_start_date_1y"] = first
            audit[f"{variable}_macro_window_end_date_1y"] = last
            audit[f"{variable}_macro_window_n_obs_1y"] = count
            audit[f"{variable}_macro_window_complete_1y"] = int(complete)
        # Requested shorter name; the full variable family is also retained.
        if "unemployment_rate" in variable_definitions:
            record["unemployment_change_1y"] = record["unemployment_rate_change_1y"]
        records.append(record)
        audits.append(audit)
    index = pd.DatetimeIndex(prediction_dates).astype("datetime64[ns]")
    features = pd.DataFrame(records, index=index)
    audit_frame = pd.DataFrame(audits, index=index)
    audit_frame = pd.DataFrame({
        column: (pd.to_datetime(audit_frame[column]).dt.normalize().astype("datetime64[ns]")
                 if "_date" in column else audit_frame[column])
        for column in audit_frame
    }, index=index)
    references = [f"{variable}_macro_reference_date" for variable in variable_definitions]
    audit_frame["macro_feature_reference_date"] = audit_frame[references].max(axis=1)
    return features, audit_frame

unique_dates = pd.DatetimeIndex(event_dates.unique()).sort_values()
features_by_date, audits_by_date = calculate_macro_features(unique_dates, histories, VARIABLES, MAX_AGE_DAYS, MIN_YEAR_OBSERVATIONS)
macro_feature_values = features_by_date.reindex(pd.DatetimeIndex(event_dates)).reset_index(drop=True)
macro_feature_audits = audits_by_date.reindex(pd.DatetimeIndex(event_dates)).reset_index(drop=True)
compact_macro_sources = {
    "inflation": "inflation_level_at_prediction_date",
    "unemployment": "unemployment_rate_level_at_prediction_date",
    "interest_rate": "federal_funds_rate_level_at_prediction_date",
    "credit_spread": "credit_spread_level_at_prediction_date",
}
for compact_name, source_name in compact_macro_sources.items():
    if source_name in macro_feature_values:
        macro_feature_values[compact_name] = macro_feature_values[source_name]
    else:
        macro_feature_values[compact_name] = np.nan
if (set(macro_feature_values.columns) | set(macro_feature_audits.columns)) & set(market_features.columns):
    raise ValueError("New macro columns conflict with existing input columns")
macro_features = pd.concat([market_features.copy(), macro_feature_values, macro_feature_audits], axis=1)
family_columns = [f"{variable}_{suffix}" for variable in VARIABLES for suffix in SUFFIXES]
macro_summary = pd.DataFrame([
    {"variable": variable, "source": spec["source"] or "not collected as a spread", "frequency": spec["frequency"],
     "level_available": int(macro_features[f"{variable}_level_at_prediction_date"].notna().sum()),
     "change_1y_available": int(macro_features[f"{variable}_change_1y"].notna().sum()),
     "change_2y_available": int(macro_features[f"{variable}_change_2y"].notna().sum()),
     "change_3y_available": int(macro_features[f"{variable}_change_3y"].notna().sum()),
     "annual_statistics_available": int(macro_features[f"{variable}_average_1y"].notna().sum())}
    for variable, spec in VARIABLES.items()
])


In [25]:
# Verify per-event alignment, endpoint dates, windows, units, and missing sources.
for variable, spec in VARIABLES.items():
    level = macro_features[f"{variable}_level_at_prediction_date"]
    reference = macro_features[f"{variable}_macro_reference_date"]
    available = level.notna()
    for lag in [1, 2, 3]:
        past_date = macro_features[f"{variable}_macro_reference_date_{lag}y"]
        cutoff = macro_features[f"{variable}_macro_lag_cutoff_date_{lag}y"]
        observed = past_date.notna()
    first = macro_features[f"{variable}_macro_window_start_date_1y"]
    last = macro_features[f"{variable}_macro_window_end_date_1y"]
    observed_window = first.notna()
    complete = macro_features[f"{variable}_macro_window_complete_1y"].astype(bool)
    volatility = macro_features[f"{variable}_volatility_1y"]
for column in macro_feature_audits:
    if "_date" in column:
        reference = macro_feature_audits[column]
        observed = reference.notna()

output_path = PROCESSED / "macro_features.csv"
macro_features.to_csv(output_path, index=False, date_format="%Y-%m-%d")
